# flattened_hamiltonian_analysis

Standalone notebook for the OW flattened-Hamiltonian benchmark.

This notebook reproduces the flattened-H analysis from [exact_DW_GS_benchmark.ipynb](../exact_DW_GS_benchmark.ipynb) and extends it to:
- the DW benchmark case with `alpha_1 = 1` as the topological-region mass and `alpha_2 = 30` as the trivial-region mass,
- a DW trivial-check case with `alpha_1 = 3`, `alpha_2 = 30`,
- exactly the `nshell = None, 0, 1, 2` characterizations for each DW case.

Assumption:
The convention from this notebook forward is `alpha_1 = topological-region mass` and `alpha_2 = trivial-region mass`; the DW case therefore uses `DW=True`, `alpha_1=1`, `alpha_2=30`; the extra DW trivial-check case uses `alpha_1=3`, `alpha_2=30`.


## Colab Bootstrap

If you want to run the GPU simulation and CPU characterization from Google Colab with a 40 GB A100, use the following setup cells before running the scripts.

```python
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
```

```python
!pip install -q h5py pandas matplotlib tqdm pillow
```

```python
%cd /content
!git clone <your_repo_url>
%cd /content/class_A_fermionic_adaptive_circuit
```

Optional persistent storage:

```python
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/classA_outputs
```

GPU simulation:

```python
!python gpu_sims/run_perfect_correction_dw_sim_gpu.py \
  --device cuda:0 \
  --dtype complex128 \
  --output-root /content/drive/MyDrive/classA_outputs/perfect_correction_dw_characterization \
  --samples 100 \
  --cycles 100 \
  --save-cycles 10 20 50 100 \
  --nshell-values 1 2 None \
  --triv-region-local-mode-options false true \
  --dw-truncation-options false true \
  --resume
```

CPU characterization:

```python
!python gpu_sims/run_perfect_correction_dw_characterize_cpu.py \
  --input-root /content/drive/MyDrive/classA_outputs/perfect_correction_dw_characterization \
  --fig-root /content/drive/MyDrive/classA_outputs/perfect_correction_dw_characterization_figs \
  --nshell-values 1 2 None \
  --triv-region-local-mode-options false true \
  --dw-truncation-options false true \
  --analyses entanglement correlators modular
```

Notes:
- `alpha_1=1` is the topological slab mass and `alpha_2=30` is the trivial outside mass.
- The GPU script is restartable and only saves raw sample covariances at cycles `10,20,50,100`.
- The CPU script reproduces the notebook-matched entanglement, correlator, and modular analyses from those checkpoints.


## Runtime Controls

In [ ]:

# CPU / BLAS thread controls
# Run this before any non-stdlib imports in the notebook, especially numpy/scipy/matplotlib.
import os
import multiprocessing as mp

CPU_CAP = 14
AFFINITY_START = 97  # Set to None to skip affinity pinning.

cpu_available = mp.cpu_count()
cpu_use = max(1, min(int(CPU_CAP), cpu_available))

os.environ["MY_CPU_COUNT"] = str(cpu_use)
for _k in (
    "OMP_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "MKL_NUM_THREADS",
    "VECLIB_MAXIMUM_THREADS",
    "NUMEXPR_NUM_THREADS",
    "NUMEXPR_MAX_THREADS",
):
    os.environ[_k] = str(cpu_use)

if AFFINITY_START is not None:
    try:
        os.sched_setaffinity(0, set(range(int(AFFINITY_START), int(AFFINITY_START) + cpu_use)))
        print(f"CPU affinity set to cores [{AFFINITY_START}, {AFFINITY_START + cpu_use - 1}]")
    except Exception as exc:
        print(f"CPU affinity not set: {exc}")

print(f"CPU_CAP={CPU_CAP}, cpu_available={cpu_available}, cpu_use={cpu_use}")


## Setup

In [ ]:

# Minimal path setup for imports from `src`
import sys
from pathlib import Path


def find_repo_root(start):
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src" / "fgtn" / "classA_U1FGTN.py").exists():
            return candidate
    raise FileNotFoundError(f"Could not find repo root from {start}")


ROOT = find_repo_root(Path.cwd())
NOTEBOOK_ANALYSIS_DIR = ROOT / "notebooks" / "flattened_hamiltonian_analysis"
NOTEBOOK_DATA_DIR = NOTEBOOK_ANALYSIS_DIR / "data"

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print(f"ROOT = {ROOT}")
print(f"NOTEBOOK_DATA_DIR = {NOTEBOOK_DATA_DIR}")


## Shared Imports, Helpers, and Analysis Routines

In [ ]:

import importlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

classA_U1FGTN_module = importlib.import_module("fgtn.classA_U1FGTN")
importlib.reload(classA_U1FGTN_module)
classA_U1FGTN = classA_U1FGTN_module.classA_U1FGTN

np.set_printoptions(precision=5, suppress=True)
plt.rcParams["figure.dpi"] = 120

TRIAL_ORBITALS_LIST = ["X"]
TRIAL_COLORS = {"exact": "black", "X": "tab:blue"}
TRIAL_MARKERS = {"exact": "D", "X": "o"}
SLOPE_FIT_RECORDS = []


def contiguous_y_block(ny, length, start=0):
    start = int(start) % ny
    length = int(length)
    if length < 1 or length > ny:
        raise ValueError(f"length must lie in [1, {ny}], got {length}")
    return (np.arange(length, dtype=int) + start) % ny


def contiguous_x_block(nx, start, stop_inclusive):
    start = int(start) % nx
    stop_inclusive = int(stop_inclusive) % nx
    if start <= stop_inclusive:
        return np.arange(start, stop_inclusive + 1, dtype=int)
    return np.concatenate([
        np.arange(start, nx, dtype=int),
        np.arange(0, stop_inclusive + 1, dtype=int),
    ])


def subsystem_indices(nx, ny, x_vals, y_vals):
    x_vals = np.asarray(x_vals, dtype=int)
    y_vals = np.asarray(y_vals, dtype=int)
    idx = []
    for y in y_vals:
        base_y = 2 * nx * int(y)
        for x in x_vals:
            x = int(x) % nx
            idx.append(base_y + 2 * x)
            idx.append(base_y + 2 * x + 1)
    return np.asarray(idx, dtype=int)


def reduced_covariance(G_top, idx):
    return np.asarray(G_top, dtype=np.complex128)[np.ix_(idx, idx)]


def contour_for_subsystem(model, G_top, x_vals, y_vals):
    idx = subsystem_indices(model.Nx, model.Ny, x_vals, y_vals)
    G_sub = reduced_covariance(G_top, idx)
    s_map = model.entanglement_contour(G_sub, len(x_vals), len(y_vals))
    return G_sub, s_map


def frobenius_norm(M):
    M = np.asarray(M)
    return float(np.sqrt(np.sum(np.abs(M) ** 2)))


def unflatten_to_G6(Gfull, Nx, Ny, top=True):
    if top:
        Nlayer = 2 * Nx * Ny
        if Gfull.shape != (Nlayer, Nlayer):
            raise ValueError(f"Expected ({Nlayer}, {Nlayer}), got {Gfull.shape}")
    G6m = np.asarray(Gfull, dtype=np.complex128).reshape((2, Nx, Ny, 2, Nx, Ny), order="F")
    return np.transpose(G6m, (1, 2, 0, 4, 5, 3))


def _flatten_full_matrix(G6):
    G6 = np.asarray(G6, dtype=np.complex128)
    Nx, Ny, s1, Nx2, Ny2, s2 = G6.shape
    if not ((s1, s2) == (2, 2) and (Nx, Ny) == (Nx2, Ny2)):
        raise ValueError(f"G must have shape (Nx, Ny, 2, Nx, Ny, 2); got {G6.shape}")
    G6m = np.transpose(G6, (2, 0, 1, 5, 3, 4))
    return G6m.reshape(2 * Nx * Ny, 2 * Nx * Ny, order="F")


def _fft_y_blocks_from_flat(G6, norm="ortho", hermitize=True):
    G6 = np.asarray(G6, dtype=np.complex128)
    Nx, Ny, s1, Nx2, Ny2, s2 = G6.shape
    if not ((s1, s2) == (2, 2) and (Nx, Ny) == (Nx2, Ny2)):
        raise ValueError(f"G must have shape (Nx, Ny, 2, Nx, Ny, 2); got {G6.shape}")

    Gk6 = np.fft.fft(G6, axis=1, norm=norm)
    Gk6 = np.fft.ifft(Gk6, axis=4, norm=norm)

    G_flat = _flatten_full_matrix(Gk6)
    if hermitize:
        G_flat = 0.5 * (G_flat + G_flat.conj().T)

    blocks = []
    for j in range(Ny):
        lo = 2 * j * Nx
        hi = 2 * (j + 1) * Nx
        Bj = G_flat[lo:hi, lo:hi]
        if hermitize:
            Bj = 0.5 * (Bj + Bj.conj().T)
        blocks.append(Bj)

    ky_vals = 2 * np.pi * np.fft.fftfreq(Ny, d=1.0)
    return ky_vals, blocks


def reference_dw_locations(nx):
    half = nx // 2
    w = max(1, int(np.floor(0.2 * nx)))
    x0 = max(0, half - w)
    x1 = min(nx, half + w + 1) - 1
    return [int(x0), int(x1)]


def analysis_x_intervals(model):
    if getattr(model, "DW", False) and hasattr(model, "DW_loc") and model.DW_loc is not None:
        x0, x1 = [int(v) % model.Nx for v in model.DW_loc]
    else:
        x0, x1 = reference_dw_locations(model.Nx)
    return [
        (0, model.Nx - 1),
        (x0 - 1, x0 + 1),
        (x1 - 1, x1 + 1),
    ]


def case_title(case_spec):
    geom = "DW" if case_spec["DW"] else "uniform"
    return f"{case_spec['name']} ({geom}, alpha_1={case_spec['alpha_1']}, alpha_2={case_spec['alpha_2']})"


def topological_region_mask(model):
    Nx, Ny = model.Nx, model.Ny
    mask = np.zeros((Nx, Ny), dtype=bool)
    if getattr(model, "DW", False) and hasattr(model, "DW_loc") and isinstance(model.DW_loc, (list, tuple)) and len(model.DW_loc) == 2:
        xL = int(model.DW_loc[0]) % Nx
        xR = int(model.DW_loc[1]) % Nx
        if xL <= xR:
            mask[xL:xR + 1, :] = True
        else:
            mask[xL:, :] = True
            mask[:xR + 1, :] = True
    elif hasattr(model, "alpha_profile"):
        mask = np.isclose(np.real(model.alpha_profile), float(model.alpha_1))
    else:
        mask[:, :] = True
    if not np.any(mask):
        raise ValueError("empty topological-region mask")
    return mask


def apply_triv_region_local_mode_to_flat_hamiltonian(model, H_flat):
    if not getattr(model, "DW", False):
        return H_flat
    topo_mask = topological_region_mask(model)
    Nx, Ny = model.Nx, model.Ny
    Nlayer = 2 * Nx * Ny
    keep = np.zeros(Nlayer, dtype=bool)

    def idx(mu, x, y):
        return mu + 2 * x + 2 * Nx * y

    for x in range(Nx):
        for y in range(Ny):
            if topo_mask[x, y]:
                keep[idx(0, x, y)] = True
                keep[idx(1, x, y)] = True

    H_local = np.array(H_flat, dtype=np.complex128, copy=True)
    H_local[~keep, :] = 0.0
    H_local[:, ~keep] = 0.0
    for x in range(Nx):
        for y in range(Ny):
            if not topo_mask[x, y]:
                H_local[idx(0, x, y), idx(0, x, y)] = -1.0
                H_local[idx(1, x, y), idx(1, x, y)] = -1.0
    return 0.5 * (H_local + H_local.conj().T)


def build_exact_state(model, periodic=True, triv_region_local_mode=False, include_ky=True):
    H_eq = model._domain_wall_hamiltonian(
        periodic=periodic,
        triv_region_local_mode=triv_region_local_mode,
    )
    evals_eq = np.linalg.eigvalsh(H_eq)
    G_eq = model.G_CI_domain_wall(
        periodic=periodic,
        triv_region_local_mode=triv_region_local_mode,
    )
    data = {
        "H_eq": H_eq,
        "G_eq": G_eq,
        "evals_eq": evals_eq,
        "neg_count": int(np.count_nonzero(evals_eq < 0.0)),
        "pos_count": int(np.count_nonzero(evals_eq > 0.0)),
        "spectral_gap": float(np.min(np.abs(evals_eq))),
    }
    if include_ky:
        H_eq6 = unflatten_to_G6(H_eq, model.Nx, model.Ny, top=True)
        ky_vals, ky_blocks = _fft_y_blocks_from_flat(H_eq6, norm="ortho", hermitize=True)
        ky_evals = np.empty((model.Ny, 2 * model.Nx), dtype=float)
        for j, B in enumerate(ky_blocks):
            ky_evals[j, :] = np.sort(np.real_if_close(np.linalg.eigvalsh(B)))
        data["ky_vals_pi"] = np.fft.fftshift(ky_vals) / np.pi
        data["ky_evals"] = np.fft.fftshift(ky_evals, axes=0)
    return data


def build_flattened_trial_data(model, include_ky=True, dw_truncation=False, triv_region_local_mode=False):
    Nlayer = 2 * model.Nx * model.Ny
    H_flat_by_trial = {}
    C_ow = {}
    evals_by_trial = {}
    ky_evals_by_trial = {}
    ky_vals_pi_by_trial = {}
    mean_abs_spec_by_trial = {}
    commutator_norms_by_trial = {}

    for trial in TRIAL_ORBITALS_LIST:
        model.construct_OW_projectors(nshell=model.nshell, DW=model.DW, trial_orbitals=trial, dw_truncation=dw_truncation)

        WA_p = model.WF_Ap.reshape(Nlayer, -1)
        WB_p = model.WF_Bp.reshape(Nlayer, -1)
        WA_m = model.WF_Am.reshape(Nlayer, -1)
        WB_m = model.WF_Bm.reshape(Nlayer, -1)

        P_Ap = WA_p @ WA_p.conj().T
        P_Bp = WB_p @ WB_p.conj().T
        P_Am = WA_m @ WA_m.conj().T
        P_Bm = WB_m @ WB_m.conj().T
        commutator_norms_by_trial[trial] = frobenius_norm(P_Ap @ P_Am - P_Am @ P_Ap)

        H_flat = P_Ap + P_Bp - P_Am - P_Bm
        H_flat = 0.5 * (H_flat + H_flat.conj().T)
        if triv_region_local_mode:
            H_flat = apply_triv_region_local_mode_to_flat_hamiltonian(model, H_flat)
        H_flat_by_trial[trial] = H_flat

        evals, evecs = np.linalg.eigh(H_flat)
        evals_by_trial[trial] = evals
        mean_abs_spec_by_trial[trial] = float(np.mean(np.abs(evals)))

        if include_ky:
            H_flat6 = unflatten_to_G6(H_flat, model.Nx, model.Ny, top=True)
            ky_vals_trial, ky_blocks_trial = _fft_y_blocks_from_flat(H_flat6, norm="ortho", hermitize=True)
            ky_evals_trial = np.empty((model.Ny, 2 * model.Nx), dtype=float)
            for j, B in enumerate(ky_blocks_trial):
                ky_evals_trial[j, :] = np.sort(np.real_if_close(np.linalg.eigvalsh(B)))
            ky_vals_pi_by_trial[trial] = np.fft.fftshift(ky_vals_trial) / np.pi
            ky_evals_by_trial[trial] = np.fft.fftshift(ky_evals_trial, axes=0)

        occ_mask = evals < 0.0
        if not np.any(occ_mask):
            occ_mask = np.zeros_like(evals, dtype=bool)
            occ_mask[np.argsort(evals)[: Nlayer // 2]] = True
        U_occ = evecs[:, occ_mask]
        P_occ = U_occ @ U_occ.conj().T
        G_ow = P_occ.conj()
        C_ow[trial] = 2.0 * G_ow - np.eye(Nlayer, dtype=np.complex128)

    return {
        "H_flat_by_trial": H_flat_by_trial,
        "C_ow": C_ow,
        "evals_by_trial": evals_by_trial,
        "ky_evals_by_trial": ky_evals_by_trial,
        "ky_vals_pi_by_trial": ky_vals_pi_by_trial,
        "mean_abs_spec_by_trial": mean_abs_spec_by_trial,
        "commutator_norms_by_trial": commutator_norms_by_trial,
    }


def _fit_line_with_error(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    x, y = x[mask], y[mask]
    if x.size < 3:
        return {
            "m": np.nan,
            "m_err": np.nan,
            "b": np.nan,
            "r2": np.nan,
            "x": x,
            "y": y,
            "y_fit": np.full_like(y, np.nan),
        }
    coeffs, cov = np.polyfit(x, y, 1, cov=True)
    m, b = float(coeffs[0]), float(coeffs[1])
    m_err = float(np.sqrt(cov[0, 0])) if np.isfinite(cov[0, 0]) else np.nan
    y_fit = m * x + b
    ss_res = float(np.sum((y - y_fit) ** 2))
    ss_tot = float(np.sum((y - np.mean(y)) ** 2))
    r2 = np.nan if ss_tot == 0 else (1.0 - ss_res / ss_tot)
    return {"m": m, "m_err": m_err, "b": b, "r2": r2, "x": x, "y": y, "y_fit": y_fit}




def slope_target_for_interval(model, interval):
    x_interval = contiguous_x_block(model.Nx, interval[0], interval[1])
    if x_interval.size == model.Nx:
        return 1.0 / 3.0, "full_x"
    return 1.0 / 6.0, "wall_window"


def append_slope_fit_records(
    records,
    model,
    case_spec,
    contour_data,
    nshell,
    triv_region_local_mode=False,
    dw_truncation=False,
    configuration_label=None,
):
    for interval in contour_data["x_intervals"]:
        x_interval = contour_data["results"]["exact"][interval]["x_interval"]
        target_slope, interval_kind = slope_target_for_interval(model, interval)
        x_interval_label = f"[{int(x_interval[0])},{int(x_interval[-1])}]"
        for run_name, _ in contour_data["all_runs"]:
            state_type = "equilibrium" if run_name == "exact" else "ow_flattened"
            trial_orbital = None if run_name == "exact" else run_name
            for branch in ("lower", "upper"):
                fit = contour_data["fit_results"][run_name][interval][branch]
                slope = float(fit["m"])
                if np.isfinite(slope) and target_slope != 0:
                    percent_diff = 100.0 * abs(slope - target_slope) / abs(target_slope)
                else:
                    percent_diff = np.nan
                records.append(
                    {
                        "percent_diff": percent_diff,
                        "slope": slope,
                        "target_slope": target_slope,
                        "branch": branch,
                        "run_name": run_name,
                        "state_type": state_type,
                        "trial_orbital": trial_orbital,
                        "case_key": case_spec["key"],
                        "case_name": case_spec["name"],
                        "configuration_label": configuration_label or case_spec["key"],
                        "DW": bool(case_spec["DW"]),
                        "alpha_1": case_spec["alpha_1"],
                        "alpha_2": case_spec["alpha_2"],
                        "nshell": "None" if nshell is None else str(int(nshell)),
                        "triv_region_local_mode": bool(triv_region_local_mode),
                        "dw_truncation": bool(dw_truncation),
                        "x_interval": x_interval_label,
                        "x_interval_kind": interval_kind,
                        "x_interval_start": int(x_interval[0]),
                        "x_interval_stop": int(x_interval[-1]),
                        "r2": float(fit["r2"]),
                        "slope_err": float(fit["m_err"]),
                        "intercept": float(fit["b"]),
                        "Nx": int(model.Nx),
                        "Ny": int(model.Ny),
                    }
                )


def slope_fit_records_dataframe(records=None):
    df = pd.DataFrame(SLOPE_FIT_RECORDS if records is None else records)
    if df.empty:
        return df
    first_cols = [
        "percent_diff",
        "slope",
        "target_slope",
        "branch",
        "run_name",
        "state_type",
        "trial_orbital",
        "case_key",
        "case_name",
        "configuration_label",
        "nshell",
        "triv_region_local_mode",
        "dw_truncation",
        "x_interval",
        "x_interval_kind",
    ]
    remaining = [col for col in df.columns if col not in first_cols]
    return df[first_cols + remaining].sort_values(
        ["percent_diff", "case_key", "configuration_label", "nshell", "run_name", "x_interval", "branch"],
        na_position="last",
    ).reset_index(drop=True)

def compute_integrated_contour_results(model, G_eq, C_ow):
    x_intervals = analysis_x_intervals(model)
    Ay_values = np.arange(2, model.Ny - 1, dtype=int)
    x_full = np.arange(model.Nx, dtype=int)
    all_runs = [("exact", G_eq)] + [(trial, C_ow[trial]) for trial in TRIAL_ORBITALS_LIST]

    results = {run_name: {} for run_name, _ in all_runs}
    for run_name, C_run in all_runs:
        for x_start, x_stop_inclusive in x_intervals:
            x_interval = contiguous_x_block(model.Nx, x_start, x_stop_inclusive)
            curve = np.zeros_like(Ay_values, dtype=float)
            for i, Ay in enumerate(Ay_values):
                y_region = np.arange(Ay, dtype=int)
                _, s_map = contour_for_subsystem(model, C_run, x_full, y_region)
                curve[i] = float(np.sum(s_map[x_interval, :]))
            results[run_name][(x_start, x_stop_inclusive)] = {
                "x_interval": x_interval,
                "curve": curve,
            }

    mid = model.Ny / 2
    mask_lower = Ay_values < mid
    mask_upper = Ay_values > mid
    x_lower = np.log(np.sin(np.pi * Ay_values[mask_lower] / model.Ny))
    x_upper = np.log(np.sin(np.pi * Ay_values[mask_upper] / model.Ny))

    fit_results = {run_name: {} for run_name, _ in all_runs}
    for interval in x_intervals:
        for run_name, _ in all_runs:
            curve = results[run_name][interval]["curve"]
            fit_results[run_name][interval] = {
                "lower": _fit_line_with_error(x_lower, curve[mask_lower]),
                "upper": _fit_line_with_error(x_upper, curve[mask_upper]),
            }

    return {
        "Ay_values": Ay_values,
        "all_runs": all_runs,
        "x_intervals": x_intervals,
        "results": results,
        "fit_results": fit_results,
    }


def exact_ground_state_chern_data(case_spec, periodic=True, triv_region_local_mode=False):
    model = classA_U1FGTN(
        case_spec["Nx"],
        case_spec["Ny"],
        nshell=None,
        DW=case_spec["DW"],
        alpha_1=case_spec["alpha_1"],
        alpha_2=case_spec["alpha_2"],
    )
    exact_data = build_exact_state(
        model,
        periodic=periodic,
        triv_region_local_mode=triv_region_local_mode,
        include_ky=False,
    )
    chern_map = model.local_chern_marker_flat(exact_data["G_eq"], apply_tanh=True)
    G_top = np.asarray(exact_data["G_eq"], dtype=np.complex128)
    G_full = model._block_diag2(G_top, np.zeros_like(G_top))
    chern_number = float(np.real(model.real_space_chern_number(G_full)))
    return {
        "model": model,
        "exact_data": exact_data,
        "chern_map": chern_map,
        "chern_number": chern_number,
    }


def plot_exact_gs_chern_marker_grid(case_specs, periodic=True, triv_region_local_mode=False, ncols=2):
    case_data = [
        exact_ground_state_chern_data(
            case_spec,
            periodic=periodic,
            triv_region_local_mode=triv_region_local_mode,
        )
        for case_spec in case_specs
    ]

    ncases = len(case_data)
    ncols = max(1, min(int(ncols), ncases))
    nrows = int(np.ceil(ncases / ncols))
    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(6.0 * ncols, 4.8 * nrows),
        constrained_layout=True,
    )
    axes = np.atleast_1d(axes).ravel()

    for ax, case_spec, data in zip(axes, case_specs, case_data):
        im = ax.imshow(
            data["chern_map"].T,
            origin="lower",
            cmap="RdBu_r",
            aspect="auto",
            vmin=-1.0,
            vmax=1.0,
        )
        ax.set_title(case_title(case_spec))
        ax.set_xlabel("x")
        ax.set_ylabel("y")
        ax.set_xticks(np.arange(0, data["model"].Nx, 2))
        ax.set_yticks(np.arange(0, data["model"].Ny, 4))
        ax.set_xticks(np.arange(-0.5, data["model"].Nx, 1), minor=True)
        ax.set_yticks(np.arange(-0.5, data["model"].Ny, 1), minor=True)
        ax.grid(which="minor", color="white", linestyle="-", linewidth=0.4, alpha=0.25)
        ax.tick_params(which="minor", bottom=False, left=False)
        dw_loc = getattr(data["model"], "DW_loc", None)
        textbox = (
            rf"$N_x={data['model'].Nx},\ N_y={data['model'].Ny}$" + "\n"
            + rf"$\alpha_1={data['model'].alpha_1},\ \alpha_2={data['model'].alpha_2}$" + "\n"
            + (rf"DWs at $x={dw_loc}$" + "\n" if dw_loc is not None else "")
            + rf"$C_{{\mathrm{{rs}}}} = {data['chern_number']:.5f}$"
        )
        ax.text(
            0.03,
            0.97,
            textbox,
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=9,
            linespacing=1.35,
            bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="black", alpha=0.85),
        )

    for ax in axes[ncases:]:
        ax.axis("off")

    fig.colorbar(im, ax=axes[:ncases].tolist(), label="tanh(C(r))", shrink=0.92)
    fig.suptitle("Exact-ground-state local Chern marker across benchmark cases")
    plt.show()

    print("Exact GS real-space Chern numbers")
    print("===============================")
    for case_spec, data in zip(case_specs, case_data):
        print(f"{case_title(case_spec)}: C_rs = {data['chern_number']:.5f}")

    return case_data

def plot_flattened_spectra(model, trial_data, case_spec, include_ky=True):
    nshell_label = "None" if model.nshell is None else str(model.nshell)
    if include_ky:
        fig, axes = plt.subplots(2, len(TRIAL_ORBITALS_LIST), figsize=(5.5 * len(TRIAL_ORBITALS_LIST), 8.0), constrained_layout=True, squeeze=False)
        for col, trial in enumerate(TRIAL_ORBITALS_LIST):
            ax = axes[0, col]
            ax.plot(trial_data["evals_by_trial"][trial], ".", ms=2, color=TRIAL_COLORS[trial])
            ax.set_title(rf"Flattened $H$, $\tau \in \sigma_{{{trial}}}$")
            ax.set_xlabel("eigenvalue index")
            ax.set_ylabel("energy")
            ax.grid(alpha=0.3)

            ax = axes[1, col]
            for band in trial_data["ky_evals_by_trial"][trial].T:
                ax.plot(trial_data["ky_vals_pi_by_trial"][trial], band, ".", ms=3, color=TRIAL_COLORS[trial])
            ax.set_title(rf"Flattened $H$ vs $k_y/\pi$, $\tau \in \sigma_{{{trial}}}$")
            ax.set_xlabel(r"$k_y/\pi$")
            ax.set_ylabel("energy")
            ax.set_xlim(-1.0, 1.0)
            ax.set_xticks([-1.0, -0.5, 0.0, 0.5, 1.0])
            ax.grid(alpha=0.3)
        fig.suptitle(f"Flattened Hamiltonian spectra | {case_title(case_spec)} | nshell={nshell_label}")
        plt.show()
    else:
        fig, axes = plt.subplots(1, len(TRIAL_ORBITALS_LIST), figsize=(5.5 * len(TRIAL_ORBITALS_LIST), 4.0), constrained_layout=True)
        axes = np.atleast_1d(axes)
        for ax, trial in zip(axes, TRIAL_ORBITALS_LIST):
            ax.plot(trial_data["evals_by_trial"][trial], ".", ms=2, color=TRIAL_COLORS[trial])
            ax.set_title(rf"Flattened $H$, $\tau \in \sigma_{{{trial}}}$")
            ax.set_xlabel("eigenvalue index")
            ax.set_ylabel("energy")
            ax.grid(alpha=0.3)
        fig.suptitle(f"Flattened Hamiltonian spectra | {case_title(case_spec)} | nshell={nshell_label}")
        plt.show()

    fig, axes = plt.subplots(1, len(TRIAL_ORBITALS_LIST), figsize=(5.5 * len(TRIAL_ORBITALS_LIST), 4.0), constrained_layout=True)
    axes = np.atleast_1d(axes)
    for ax, trial in zip(axes, TRIAL_ORBITALS_LIST):
        ax.hist(trial_data["evals_by_trial"][trial], bins=60, color=TRIAL_COLORS[trial], alpha=0.85)
        ax.set_title(rf"Flattened $H$ histogram, $\tau \in \sigma_{{{trial}}}$")
        ax.set_xlabel("energy")
        ax.set_ylabel("count")
        ax.grid(alpha=0.3)
    fig.suptitle(f"Flattened Hamiltonian eigenvalue histograms | {case_title(case_spec)} | nshell={nshell_label}")
    plt.show()


def print_flattened_summary(model, exact_data, trial_data, case_spec):
    print(case_title(case_spec))
    print(f"  nshell={model.nshell}")
    print(f"  neg_count={exact_data['neg_count']}, pos_count={exact_data['pos_count']}, min |E_eq|={exact_data['spectral_gap']:.6e}")
    for trial in TRIAL_ORBITALS_LIST:
        print(
            f"  trial {trial}: mean(abs(spec(H_flat)))={trial_data['mean_abs_spec_by_trial'][trial]:.6f}, "
            f"||[P_A^+, P_A^-]||_F={trial_data['commutator_norms_by_trial'][trial]:.6e}"
        )


def plot_integrated_contour_vs_Ay(model, contour_data, case_spec):
    n_runs = len(contour_data["all_runs"])
    fig, axes = plt.subplots(1, n_runs, figsize=(5.5 * n_runs, 4.5), constrained_layout=True)
    axes = np.atleast_1d(axes)
    interval_colors = plt.cm.viridis(np.linspace(0.08, 0.92, len(contour_data["x_intervals"])))
    dw_loc = getattr(model, "DW_loc", None)
    if dw_loc is None:
        dw_loc = reference_dw_locations(model.Nx)

    for ax, (run_name, _) in zip(axes, contour_data["all_runs"]):
        for color, interval in zip(interval_colors, contour_data["x_intervals"]):
            x_interval = contour_data["results"][run_name][interval]["x_interval"]
            curve = contour_data["results"][run_name][interval]["curve"]
            label = rf"$x \in [{int(x_interval[0])}, {int(x_interval[-1])}]$"
            ax.plot(contour_data["Ay_values"], curve, marker="o", ms=3, lw=1.5, color=color, label=label)
        ax.set_title(
            r"Exact $G_{\mathrm{eq}}$" if run_name == "exact"
            else rf"OW flattened $H$, $\tau \in \sigma_{{{run_name}}}$"
        )
        ax.set_xlabel(r"$A_y$")
        ax.set_ylabel(r"$\sum_{x \in I_x}\sum_{y \in A} s_A(x,y)$")
        ax.grid(alpha=0.3)
        ax.legend(title="x interval", fontsize=8)

    textbox = (
        rf"$N_x={model.Nx},\ N_y={model.Ny}$" + "\n" +
        rf"$\alpha_1={model.alpha_1},\ \alpha_2={model.alpha_2}$" + "\n" +
        rf"reference walls at $x={dw_loc}$"
    )
    axes[0].text(
        0.03,
        0.55,
        textbox,
        transform=axes[0].transAxes,
        ha="left",
        va="top",
        fontsize=9,
        linespacing=1.4,
        bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="black", alpha=0.85),
    )
    fig.suptitle(f"Integrated contour vs $A_y$ | {case_title(case_spec)} | nshell={model.nshell}")
    plt.show()


def plot_log_chord_fits(model, contour_data, case_spec):
    fig, axes = plt.subplots(1, len(contour_data["x_intervals"]), figsize=(7.0 * len(contour_data["x_intervals"]), 5.0), constrained_layout=True)
    axes = np.atleast_1d(axes)
    for ax, interval in zip(axes, contour_data["x_intervals"]):
        x_interval = contiguous_x_block(model.Nx, interval[0], interval[1])
        ax.set_title(rf"$x \in [{int(x_interval[0])},\, {int(x_interval[-1])}]$")
        ax.set_xlabel(r"$\log\!\left[\sin\!\left(\pi A_y/N_y\right)\right]$")
        ax.set_ylabel(r"$\sum_{x \in I_x}\sum_{y \in A} s_A(x,y)$")
        ax.grid(alpha=0.3)

        for run_name, _ in contour_data["all_runs"]:
            color = TRIAL_COLORS[run_name]
            marker = TRIAL_MARKERS[run_name]
            lower_fit = contour_data["fit_results"][run_name][interval]["lower"]
            upper_fit = contour_data["fit_results"][run_name][interval]["upper"]
            run_label = "exact" if run_name == "exact" else rf"$\sigma_{{{run_name}}}$"

            ax.plot(lower_fit["x"], lower_fit["y"], marker, ms=3, color=color, alpha=0.85,
                    label=(run_label
                           + rf" lower: $m={lower_fit['m']:.4f}\pm{lower_fit['m_err']:.2e}$"
                           + rf", $R^2={lower_fit['r2']:.4f}$"))
            ax.plot(lower_fit["x"], lower_fit["y_fit"], "-", lw=1.4, color=color)

            ax.plot(upper_fit["x"], upper_fit["y"], marker, ms=3, color=color, alpha=0.35,
                    label=(run_label
                           + rf" upper: $m={upper_fit['m']:.4f}\pm{upper_fit['m_err']:.2e}$"
                           + rf", $R^2={upper_fit['r2']:.4f}$"))
            ax.plot(upper_fit["x"], upper_fit["y_fit"], "--", lw=1.4, color=color, alpha=0.5)

        ax.legend(fontsize=7, ncol=1)

    fig.suptitle(f"Log-chord fits | {case_title(case_spec)} | nshell={model.nshell}")
    plt.show()

    print("Fit summary")
    print("===========")
    for interval in contour_data["x_intervals"]:
        x_interval = contiguous_x_block(model.Nx, interval[0], interval[1])
        print(f"x interval {interval} -> x = [{int(x_interval[0])}, ..., {int(x_interval[-1])}]")
        print(f"  {'run':<8}  {'branch':<7}  {'slope':>10}  {'slope_err':>12}  {'intercept':>12}  {'R2':>8}")
        print(f"  {'-'*8}  {'-'*7}  {'-'*10}  {'-'*12}  {'-'*12}  {'-'*8}")
        for run_name, _ in contour_data["all_runs"]:
            for branch in ("lower", "upper"):
                r = contour_data["fit_results"][run_name][interval][branch]
                print(f"  {run_name:<8}  {branch:<7}  {r['m']:>+10.6f}  {r['m_err']:>12.4e}  {r['b']:>12.6f}  {r['r2']:>8.6f}")
    print(f"A_y scan = {contour_data['Ay_values'][0]} ... {contour_data['Ay_values'][-1]}")


def run_case_characterization(
    case_spec,
    nshell=None,
    periodic=True,
    triv_region_local_mode=False,
    include_ky=True,
    run_entanglement_characterizations=True,
    dw_truncation=False,
    record_slope_fits=True,
    configuration_label=None,
):
    model = classA_U1FGTN(
        case_spec["Nx"],
        case_spec["Ny"],
        nshell=nshell,
        DW=case_spec["DW"],
        alpha_1=case_spec["alpha_1"],
        alpha_2=case_spec["alpha_2"],
    )
    exact_data = build_exact_state(model, periodic=periodic, triv_region_local_mode=triv_region_local_mode, include_ky=include_ky)
    trial_data = build_flattened_trial_data(
        model,
        include_ky=include_ky,
        dw_truncation=dw_truncation,
        triv_region_local_mode=triv_region_local_mode,
    )

    plot_flattened_spectra(model, trial_data, case_spec, include_ky=include_ky)
    print_flattened_summary(model, exact_data, trial_data, case_spec)

    contour_data = None
    if run_entanglement_characterizations:
        contour_data = compute_integrated_contour_results(model, exact_data["G_eq"], trial_data["C_ow"])
        plot_integrated_contour_vs_Ay(model, contour_data, case_spec)
        plot_log_chord_fits(model, contour_data, case_spec)
        if record_slope_fits:
            append_slope_fit_records(
                SLOPE_FIT_RECORDS,
                model,
                case_spec,
                contour_data,
                nshell=nshell,
                triv_region_local_mode=triv_region_local_mode,
                dw_truncation=dw_truncation,
                configuration_label=configuration_label,
            )


    return {
        "model": model,
        "exact_data": exact_data,
        "trial_data": trial_data,
        "contour_data": contour_data,
    }


def run_case_nshell_sweep(
    case_spec,
    nshell_values,
    periodic=True,
    triv_region_local_mode=False,
    run_entanglement_characterizations=True,
    dw_truncation=False,
    record_slope_fits=True,
    configuration_label=None,
):
    sweep_results = {}
    for nshell in nshell_values:
        print("\n" + "=" * 100)
        print(f"Running {case_title(case_spec)} | finite nshell={nshell}")
        print("=" * 100)
        sweep_results[int(nshell)] = run_case_characterization(
            case_spec,
            nshell=int(nshell),
            periodic=periodic,
            triv_region_local_mode=triv_region_local_mode,
            include_ky=False,
            run_entanglement_characterizations=run_entanglement_characterizations,
            dw_truncation=dw_truncation,
            record_slope_fits=record_slope_fits,
            configuration_label=configuration_label,
        )
    return sweep_results

## Configuration

In [ ]:

NX, NY = 20, 40
PERIODIC = True
TRIV_REGION_LOCAL_MODE = False
FINITE_NSHELLS = [0, 1, 2]


CASE_SPECS = [
    {
        "key": "dw_benchmark",
        "name": "DW benchmark",
        "DW": True,
        "alpha_1": 1,
        "alpha_2": 30,
        "Nx": NX,
        "Ny": NY,
    },
    {
        "key": "dw_trivial_check",
        "name": "DW trivial check",
        "DW": True,
        "alpha_1": 3,
        "alpha_2": 30,
        "Nx": NX,
        "Ny": NY,
    },
]

for case in CASE_SPECS:
    print(case_title(case))
print(f"finite nshell sweep = {FINITE_NSHELLS}")


## Exact-GS Chern Marker Overview

Exact local Chern marker and real-space Chern number for each alpha case, arranged in a shared subplot grid.

In [ ]:

exact_gs_chern_overview = plot_exact_gs_chern_marker_grid(
    CASE_SPECS,
    periodic=PERIODIC,
    triv_region_local_mode=TRIV_REGION_LOCAL_MODE,
    ncols=2,
)


## DW Exact-GS Chern Marker Overview with `triv_region_local_mode = True`

Exact local Chern marker and real-space Chern number for each DW case when the trivial region is replaced by decoupled onsite modes.

In [ ]:

dw_triv_local_chern_overview = plot_exact_gs_chern_marker_grid(
    [case for case in CASE_SPECS if case["DW"]],
    periodic=PERIODIC,
    triv_region_local_mode=True,
    ncols=2,
)


## Flattened-H Ground-State Chern Marker Overview

Repeat the Chern-marker characterization for the OW flattened-H ground states. Each configuration gets one grid: rows are `nshell = None, 0, 1, 2`, the column is the `X` trial orbital, and each panel includes the real-space Chern number of that flattened-H ground state.

In [ ]:

FLATTENED_CHERN_NSHELL_VALUES = [None, *FINITE_NSHELLS]
FLATTENED_CHERN_PLOT_GRIDS = True
FLATTENED_CHERN_USE_EXACT_MARKER_SIGN = True

FLATTENED_CHERN_DW_VARIANTS = [
    {"variant": "standard", "triv_region_local_mode": False, "dw_truncation": False},
    {"variant": "triv_region_local_mode", "triv_region_local_mode": True, "dw_truncation": False},
    {"variant": "dw_truncation", "triv_region_local_mode": False, "dw_truncation": True},
    {"variant": "dw_truncation_and_triv_region_local_mode", "triv_region_local_mode": True, "dw_truncation": True},
]
FLATTENED_CHERN_UNIFORM_VARIANTS = [
    {"variant": "standard", "triv_region_local_mode": False, "dw_truncation": False},
]


def flattened_chern_variants_for_case(case_spec):
    return FLATTENED_CHERN_DW_VARIANTS if case_spec["DW"] else FLATTENED_CHERN_UNIFORM_VARIANTS


def flattened_h_chern_configuration_data(case_spec, variant_spec, nshell_values=FLATTENED_CHERN_NSHELL_VALUES):
    rows = []
    for nshell in nshell_values:
        model = classA_U1FGTN(
            case_spec["Nx"],
            case_spec["Ny"],
            nshell=nshell,
            DW=case_spec["DW"],
            alpha_1=case_spec["alpha_1"],
            alpha_2=case_spec["alpha_2"],
        )
        trial_data = build_flattened_trial_data(
            model,
            include_ky=False,
            dw_truncation=variant_spec["dw_truncation"],
            triv_region_local_mode=variant_spec["triv_region_local_mode"],
        )
        for trial in TRIAL_ORBITALS_LIST:
            G_top = np.asarray(trial_data["C_ow"][trial], dtype=np.complex128)
            # The flattened-H entropy/correlation sections use C_ow = 2 P_-^* - I.
            # For the local Chern marker, match the exact-GS marker orientation used
            # elsewhere in this notebook; for the OW flattened Hamiltonian this is -C_ow.
            G_chern = -G_top if FLATTENED_CHERN_USE_EXACT_MARKER_SIGN else G_top
            chern_map = model.local_chern_marker_flat(G_chern, apply_tanh=True)
            G_full = model._block_diag2(G_chern, np.zeros_like(G_chern))
            chern_number = float(np.real(model.real_space_chern_number(G_full)))
            rows.append(
                {
                    "case_key": case_spec["key"],
                    "case_name": case_spec["name"],
                    "DW": bool(case_spec["DW"]),
                    "alpha_1": case_spec["alpha_1"],
                    "alpha_2": case_spec["alpha_2"],
                    "variant": variant_spec["variant"],
                    "triv_region_local_mode": bool(variant_spec["triv_region_local_mode"]),
                    "dw_truncation": bool(variant_spec["dw_truncation"]),
                    "nshell": "None" if nshell is None else str(int(nshell)),
                    "nshell_value": np.nan if nshell is None else int(nshell),
                    "trial_orbital": trial,
                    "chern_number": chern_number,
                    "chern_covariance_convention": "-C_ow" if FLATTENED_CHERN_USE_EXACT_MARKER_SIGN else "C_ow",
                    "mean_abs_spec": trial_data["mean_abs_spec_by_trial"][trial],
                    "commutator_norm": trial_data["commutator_norms_by_trial"][trial],
                    "chern_map": chern_map,
                    "DW_loc": getattr(model, "DW_loc", None),
                    "Nx": model.Nx,
                    "Ny": model.Ny,
                }
            )
    return rows


def plot_flattened_h_chern_configuration_grid(rows, case_spec, variant_spec):
    nshell_labels = ["None" if nshell is None else str(int(nshell)) for nshell in FLATTENED_CHERN_NSHELL_VALUES]
    nrows = len(nshell_labels)
    ncols = len(TRIAL_ORBITALS_LIST)
    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(4.8 * ncols, 3.8 * nrows),
        constrained_layout=True,
        squeeze=False,
    )
    row_lookup = {(row["nshell"], row["trial_orbital"]): row for row in rows}
    last_im = None
    for r, nshell_label in enumerate(nshell_labels):
        for c, trial in enumerate(TRIAL_ORBITALS_LIST):
            ax = axes[r, c]
            row = row_lookup[(nshell_label, trial)]
            last_im = ax.imshow(
                row["chern_map"].T,
                origin="lower",
                cmap="RdBu_r",
                aspect="auto",
                vmin=-1.0,
                vmax=1.0,
            )
            ax.set_title(rf"$\sigma_{{{trial}}}$, nshell={nshell_label}")
            ax.set_xlabel("x")
            ax.set_ylabel("y")
            ax.set_xticks(np.arange(0, row["Nx"], max(1, row["Nx"] // 5)))
            ax.set_yticks(np.arange(0, row["Ny"], max(1, row["Ny"] // 5)))
            ax.set_xticks(np.arange(-0.5, row["Nx"], 1), minor=True)
            ax.set_yticks(np.arange(-0.5, row["Ny"], 1), minor=True)
            ax.grid(which="minor", color="white", linestyle="-", linewidth=0.35, alpha=0.2)
            ax.tick_params(which="minor", bottom=False, left=False)
            textbox = (
                rf"$C_{{\mathrm{{rs}}}}={row['chern_number']:.5f}$" + "\n"
                + rf"$\langle |E| \rangle={row['mean_abs_spec']:.5f}$" + "\n"
                + rf"$||[P_A^+,P_A^-]||_F={row['commutator_norm']:.2e}$"
            )
            ax.text(
                0.03,
                0.97,
                textbox,
                transform=ax.transAxes,
                ha="left",
                va="top",
                fontsize=8,
                linespacing=1.25,
                bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="black", alpha=0.82),
            )
    if last_im is not None:
        fig.colorbar(last_im, ax=axes.ravel().tolist(), shrink=0.78, label="tanh-compressed local Chern marker")
    sign_label = "exact-marker sign (-C_ow)" if FLATTENED_CHERN_USE_EXACT_MARKER_SIGN else "raw C_ow sign"
    fig.suptitle(
        f"Flattened-H GS Chern markers | {case_title(case_spec)} | variant={variant_spec['variant']} | {sign_label}",
        y=1.01,
    )
    plt.show()
    return fig, axes


flattened_chern_records = []
flattened_chern_figures = {}
for case_spec in CASE_SPECS:
    for variant_spec in flattened_chern_variants_for_case(case_spec):
        config_rows = flattened_h_chern_configuration_data(case_spec, variant_spec)
        flattened_chern_records.extend(config_rows)
        config_key = (case_spec["key"], variant_spec["variant"])
        if FLATTENED_CHERN_PLOT_GRIDS:
            flattened_chern_figures[config_key] = plot_flattened_h_chern_configuration_grid(
                config_rows,
                case_spec,
                variant_spec,
            )

flattened_chern_df = pd.DataFrame([
    {k: v for k, v in row.items() if k != "chern_map"}
    for row in flattened_chern_records
])
first_cols = [
    "chern_number",
    "case_key",
    "variant",
    "nshell",
    "trial_orbital",
    "alpha_1",
    "alpha_2",
    "triv_region_local_mode",
    "dw_truncation",
    "mean_abs_spec",
    "commutator_norm",
    "chern_covariance_convention",
]
remaining_cols = [col for col in flattened_chern_df.columns if col not in first_cols]
flattened_chern_df = flattened_chern_df[first_cols + remaining_cols].sort_values(
    ["case_key", "variant", "nshell", "trial_orbital"],
    na_position="last",
).reset_index(drop=True)
flattened_chern_df


## Reset Slope-Fit Accumulator

Run this before the characterization sections if you want the final dataframe to contain one copy of each fit row.

In [ ]:

SLOPE_FIT_RECORDS.clear()


## DW benchmark: `alpha_1 = 1`, `alpha_2 = 30`

Standard DW flattened-H characterization with topological mass `alpha_1=1` and trivial mass `alpha_2=30`.

### `nshell = None` characterizations

In [ ]:

case_spec = CASE_SPECS[0]
dw_benchmark_nshell_none = run_case_characterization(
    case_spec,
    nshell=None,
    periodic=PERIODIC,
    triv_region_local_mode=False,
    include_ky=True,
    run_entanglement_characterizations=True,
    dw_truncation=False,
    configuration_label="dw_benchmark_standard",
)


### `nshell in {0,1,2}` characterizations

In [ ]:

case_spec = CASE_SPECS[0]
dw_benchmark_finite_nshell = run_case_nshell_sweep(
    case_spec,
    FINITE_NSHELLS,
    periodic=PERIODIC,
    triv_region_local_mode=False,
    run_entanglement_characterizations=True,
    dw_truncation=False,
    configuration_label="dw_benchmark_standard",
)


## DW benchmark with `triv_region_local_mode = True`

Replace the trivial region by decoupled onsite modes in both the exact-H/G construction and the OW flattened-H construction.

### `nshell = None` characterizations

In [ ]:

case_spec = CASE_SPECS[0]
dw_benchmark_triv_local_nshell_none = run_case_characterization(
    case_spec,
    nshell=None,
    periodic=PERIODIC,
    triv_region_local_mode=True,
    include_ky=True,
    run_entanglement_characterizations=True,
    dw_truncation=False,
    configuration_label="dw_benchmark_triv_local",
)


### `nshell in {0,1,2}` characterizations with `triv_region_local_mode = True`

In [ ]:

case_spec = CASE_SPECS[0]
dw_benchmark_triv_local_finite_nshell = run_case_nshell_sweep(
    case_spec,
    FINITE_NSHELLS,
    periodic=PERIODIC,
    triv_region_local_mode=True,
    run_entanglement_characterizations=True,
    dw_truncation=False,
    configuration_label="dw_benchmark_triv_local",
)


## DW benchmark with `dw_truncation = True`

Additionally truncate OW modes to their own DW sector while keeping `triv_region_local_mode = False`.

### `nshell = None` characterizations

In [ ]:

case_spec = CASE_SPECS[0]
dw_benchmark_dw_trunc_nshell_none = run_case_characterization(
    case_spec,
    nshell=None,
    periodic=PERIODIC,
    triv_region_local_mode=False,
    include_ky=True,
    run_entanglement_characterizations=True,
    dw_truncation=True,
    configuration_label="dw_benchmark_dw_trunc",
)


### `nshell in {0,1,2}` characterizations with `dw_truncation = True`

In [ ]:

case_spec = CASE_SPECS[0]
dw_benchmark_dw_trunc_finite_nshell = run_case_nshell_sweep(
    case_spec,
    FINITE_NSHELLS,
    periodic=PERIODIC,
    triv_region_local_mode=False,
    run_entanglement_characterizations=True,
    dw_truncation=True,
    configuration_label="dw_benchmark_dw_trunc",
)


## DW benchmark with `dw_truncation = True` and `triv_region_local_mode = True`

Enable both the OW DW-sector truncation and the trivial-region local-mode branch.

### `nshell = None` characterizations

In [ ]:

case_spec = CASE_SPECS[0]
dw_benchmark_dw_trunc_triv_local_nshell_none = run_case_characterization(
    case_spec,
    nshell=None,
    periodic=PERIODIC,
    triv_region_local_mode=True,
    include_ky=True,
    run_entanglement_characterizations=True,
    dw_truncation=True,
    configuration_label="dw_benchmark_dw_trunc_triv_local",
)


### `nshell in {0,1,2}` characterizations with `dw_truncation = True` and `triv_region_local_mode = True`

In [ ]:

case_spec = CASE_SPECS[0]
dw_benchmark_dw_trunc_triv_local_finite_nshell = run_case_nshell_sweep(
    case_spec,
    FINITE_NSHELLS,
    periodic=PERIODIC,
    triv_region_local_mode=True,
    run_entanglement_characterizations=True,
    dw_truncation=True,
    configuration_label="dw_benchmark_dw_trunc_triv_local",
)


## DW trivial check: `alpha_1 = 3`, `alpha_2 = 30`

DW check with a trivial slab mass `alpha_1=3` and trivial exterior mass `alpha_2=30`.

### `nshell = None` characterizations

In [ ]:

case_spec = CASE_SPECS[1]
dw_trivial_check_nshell_none = run_case_characterization(
    case_spec,
    nshell=None,
    periodic=PERIODIC,
    triv_region_local_mode=False,
    include_ky=True,
    run_entanglement_characterizations=True,
    dw_truncation=False,
    configuration_label="dw_trivial_check_standard",
)


### `nshell in {0,1,2}` characterizations

In [ ]:

case_spec = CASE_SPECS[1]
dw_trivial_check_finite_nshell = run_case_nshell_sweep(
    case_spec,
    FINITE_NSHELLS,
    periodic=PERIODIC,
    triv_region_local_mode=False,
    run_entanglement_characterizations=True,
    dw_truncation=False,
    configuration_label="dw_trivial_check_standard",
)


## DW trivial check with `triv_region_local_mode = True`

Repeat the trivial DW check with decoupled local modes in the outside region for exact H/G and OW flattened H.

### `nshell = None` characterizations

In [ ]:

case_spec = CASE_SPECS[1]
dw_trivial_check_triv_local_nshell_none = run_case_characterization(
    case_spec,
    nshell=None,
    periodic=PERIODIC,
    triv_region_local_mode=True,
    include_ky=True,
    run_entanglement_characterizations=True,
    dw_truncation=False,
    configuration_label="dw_trivial_check_triv_local",
)


### `nshell in {0,1,2}` characterizations with `triv_region_local_mode = True`

In [ ]:

case_spec = CASE_SPECS[1]
dw_trivial_check_triv_local_finite_nshell = run_case_nshell_sweep(
    case_spec,
    FINITE_NSHELLS,
    periodic=PERIODIC,
    triv_region_local_mode=True,
    run_entanglement_characterizations=True,
    dw_truncation=False,
    configuration_label="dw_trivial_check_triv_local",
)


## DW trivial check with `dw_truncation = True`

Repeat the trivial DW check with OW DW-sector truncation only.

### `nshell = None` characterizations

In [ ]:

case_spec = CASE_SPECS[1]
dw_trivial_check_dw_trunc_nshell_none = run_case_characterization(
    case_spec,
    nshell=None,
    periodic=PERIODIC,
    triv_region_local_mode=False,
    include_ky=True,
    run_entanglement_characterizations=True,
    dw_truncation=True,
    configuration_label="dw_trivial_check_dw_trunc",
)


### `nshell in {0,1,2}` characterizations with `dw_truncation = True`

In [ ]:

case_spec = CASE_SPECS[1]
dw_trivial_check_dw_trunc_finite_nshell = run_case_nshell_sweep(
    case_spec,
    FINITE_NSHELLS,
    periodic=PERIODIC,
    triv_region_local_mode=False,
    run_entanglement_characterizations=True,
    dw_truncation=True,
    configuration_label="dw_trivial_check_dw_trunc",
)


## DW trivial check with `dw_truncation = True` and `triv_region_local_mode = True`

Repeat the trivial DW check with both OW DW-sector truncation and local-mode outside region.

### `nshell = None` characterizations

In [ ]:

case_spec = CASE_SPECS[1]
dw_trivial_check_dw_trunc_triv_local_nshell_none = run_case_characterization(
    case_spec,
    nshell=None,
    periodic=PERIODIC,
    triv_region_local_mode=True,
    include_ky=True,
    run_entanglement_characterizations=True,
    dw_truncation=True,
    configuration_label="dw_trivial_check_dw_trunc_triv_local",
)


### `nshell in {0,1,2}` characterizations with `dw_truncation = True` and `triv_region_local_mode = True`

In [ ]:

case_spec = CASE_SPECS[1]
dw_trivial_check_dw_trunc_triv_local_finite_nshell = run_case_nshell_sweep(
    case_spec,
    FINITE_NSHELLS,
    periodic=PERIODIC,
    triv_region_local_mode=True,
    run_entanglement_characterizations=True,
    dw_truncation=True,
    configuration_label="dw_trivial_check_dw_trunc_triv_local",
)


## Slope-Fit Summary Dataframe

Compile every recorded integrated-contour log-chord fit into one dataframe. The first column is the absolute percent difference to the reference slope: `1/3` for the full-width interval and `1/6` for the wall-window intervals.

In [ ]:

slope_fit_df = slope_fit_records_dataframe()
slope_fit_df


## Slope-Fit Summary by Configuration

Compact view of the average percent difference grouped by configuration details.

In [ ]:

if slope_fit_df.empty:
    slope_fit_summary_df = slope_fit_df
else:
    slope_fit_summary_df = (
        slope_fit_df
        .groupby([
            "case_key",
            "configuration_label",
            "nshell",
            "triv_region_local_mode",
            "dw_truncation",
            "run_name",
            "x_interval_kind",
        ], dropna=False)
        .agg(
            mean_percent_diff=("percent_diff", "mean"),
            max_percent_diff=("percent_diff", "max"),
            mean_slope=("slope", "mean"),
            mean_r2=("r2", "mean"),
            n_fits=("slope", "count"),
        )
        .reset_index()
        .sort_values("mean_percent_diff", na_position="last")
    )
slope_fit_summary_df


# Standalone $x$-Averaged Squared-Correlator Scaling Sweep

This section is intentionally self-contained: it resets CPU/thread limits, sets up paths/imports, rebuilds exact and OW flattened-H states for every studied configuration, fits the x-averaged correlator

$$C_{\mathrm{avg}}(r_y)=\frac{1}{N_x}\sum_x C_G(x,r_y)$$

against chord length, also fits the two fixed-$x$ domain-wall slices when a DW is present, and returns one dataframe with the slope percent error relative to the chiral-free-fermion target slope $-2$.

In [ ]:

# Standalone runtime controls for this section. Run this before the import cell when using this section alone.
import os

CPU_CAP_CAVG = int(os.environ.get("CPU_CAP", "14"))
for _var in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[_var] = str(CPU_CAP_CAVG)

try:
    _available = sorted(os.sched_getaffinity(0))
    _use = set(_available[: min(CPU_CAP_CAVG, len(_available))])
    os.sched_setaffinity(0, _use)
    print(f"C_avg section CPU affinity set to {sorted(_use)}")
except Exception as exc:
    print(f"C_avg section CPU affinity unchanged: {exc}")

print({var: os.environ.get(var) for var in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS")})


In [ ]:

# Standalone path setup for this section.
import sys
from pathlib import Path


def cavg_find_repo_root(start):
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src" / "fgtn" / "classA_U1FGTN.py").exists():
            return candidate
    raise FileNotFoundError(f"Could not find repo root from {start}")


CAVG_ROOT = cavg_find_repo_root(Path.cwd())
CAVG_NOTEBOOK_ANALYSIS_DIR = CAVG_ROOT / "notebooks" / "flattened_hamiltonian_analysis"
CAVG_NOTEBOOK_DATA_DIR = CAVG_NOTEBOOK_ANALYSIS_DIR / "data"

CAVG_SRC = CAVG_ROOT / "src"
if str(CAVG_SRC) not in sys.path:
    sys.path.insert(0, str(CAVG_SRC))

print(f"C_avg ROOT = {CAVG_ROOT}")
print(f"C_avg data dir = {CAVG_NOTEBOOK_DATA_DIR}")


In [ ]:

# Standalone imports and configuration for the C_avg sweep.
import importlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

classA_U1FGTN_module_cavg = importlib.import_module("fgtn.classA_U1FGTN")
importlib.reload(classA_U1FGTN_module_cavg)
classA_U1FGTN_cavg = classA_U1FGTN_module_cavg.classA_U1FGTN

CAVG_NX, CAVG_NY = 20, 40
CAVG_PERIODIC = True
CAVG_NSHELL_VALUES = [None, 0, 1, 2]
CAVG_TRIAL_ORBITALS = ["X"]
CAVG_FIT_RY_MIN = 3
CAVG_FIT_RY_MAX = min(15, CAVG_NY // 2 - 1)
CAVG_TARGET_SLOPE = -2.0
CAVG_PLOT_PROFILES = False

CAVG_CASE_SPECS = [
    {"key": "dw_benchmark", "name": "DW benchmark", "DW": True, "alpha_1": 1, "alpha_2": 30, "Nx": CAVG_NX, "Ny": CAVG_NY},
    {"key": "dw_trivial_check", "name": "DW trivial check", "DW": True, "alpha_1": 3, "alpha_2": 30, "Nx": CAVG_NX, "Ny": CAVG_NY},
]

CAVG_DW_VARIANTS = [
    {"variant": "standard", "triv_region_local_mode": False, "dw_truncation": False},
    {"variant": "triv_region_local_mode", "triv_region_local_mode": True, "dw_truncation": False},
    {"variant": "dw_truncation", "triv_region_local_mode": False, "dw_truncation": True},
    {"variant": "dw_truncation_and_triv_region_local_mode", "triv_region_local_mode": True, "dw_truncation": True},
]

print(f"C_avg fit window: r_y in [{CAVG_FIT_RY_MIN}, {CAVG_FIT_RY_MAX}]")
for _case in CAVG_CASE_SPECS:
    print(_case)


In [ ]:

# Standalone helper functions for x-averaged squared-correlator characterization.

def cavg_case_title(case_spec):
    geom = "DW" if case_spec["DW"] else "uniform"
    return f"{case_spec['name']} ({geom}, alpha_1={case_spec['alpha_1']}, alpha_2={case_spec['alpha_2']})"


def cavg_topological_region_mask(model):
    Nx, Ny = model.Nx, model.Ny
    mask = np.zeros((Nx, Ny), dtype=bool)
    if getattr(model, "DW", False) and hasattr(model, "DW_loc") and isinstance(model.DW_loc, (list, tuple)) and len(model.DW_loc) == 2:
        xL = int(model.DW_loc[0]) % Nx
        xR = int(model.DW_loc[1]) % Nx
        if xL <= xR:
            mask[xL:xR + 1, :] = True
        else:
            mask[xL:, :] = True
            mask[:xR + 1, :] = True
    elif hasattr(model, "alpha_profile"):
        mask = np.isclose(np.real(model.alpha_profile), float(model.alpha_1))
    else:
        mask[:, :] = True
    if not np.any(mask):
        raise ValueError("empty topological-region mask")
    return mask


def cavg_apply_triv_region_local_mode_to_flat_hamiltonian(model, H_flat):
    if not getattr(model, "DW", False):
        return H_flat
    topo_mask = cavg_topological_region_mask(model)
    Nx, Ny = model.Nx, model.Ny
    Nlayer = 2 * Nx * Ny
    keep = np.zeros(Nlayer, dtype=bool)

    def idx(mu, x, y):
        return mu + 2 * x + 2 * Nx * y

    for x in range(Nx):
        for y in range(Ny):
            if topo_mask[x, y]:
                keep[idx(0, x, y)] = True
                keep[idx(1, x, y)] = True

    H_local = np.array(H_flat, dtype=np.complex128, copy=True)
    H_local[~keep, :] = 0.0
    H_local[:, ~keep] = 0.0
    for x in range(Nx):
        for y in range(Ny):
            if not topo_mask[x, y]:
                H_local[idx(0, x, y), idx(0, x, y)] = -1.0
                H_local[idx(1, x, y), idx(1, x, y)] = -1.0
    return 0.5 * (H_local + H_local.conj().T)


def cavg_build_flattened_covariances(model, *, dw_truncation=False, triv_region_local_mode=False):
    Nlayer = 2 * model.Nx * model.Ny
    covariances = {}
    spectra = {}
    for trial in CAVG_TRIAL_ORBITALS:
        model.construct_OW_projectors(
            nshell=model.nshell,
            DW=model.DW,
            trial_orbitals=trial,
            dw_truncation=dw_truncation,
        )
        WA_p = model.WF_Ap.reshape(Nlayer, -1)
        WB_p = model.WF_Bp.reshape(Nlayer, -1)
        WA_m = model.WF_Am.reshape(Nlayer, -1)
        WB_m = model.WF_Bm.reshape(Nlayer, -1)

        H_flat = WA_p @ WA_p.conj().T + WB_p @ WB_p.conj().T - WA_m @ WA_m.conj().T - WB_m @ WB_m.conj().T
        H_flat = 0.5 * (H_flat + H_flat.conj().T)
        if triv_region_local_mode:
            H_flat = cavg_apply_triv_region_local_mode_to_flat_hamiltonian(model, H_flat)

        evals, evecs = np.linalg.eigh(H_flat)
        occ_mask = evals < 0.0
        if not np.any(occ_mask):
            occ_mask = np.zeros_like(evals, dtype=bool)
            occ_mask[np.argsort(evals)[: Nlayer // 2]] = True
        P_occ = evecs[:, occ_mask] @ evecs[:, occ_mask].conj().T
        covariances[trial] = 2.0 * P_occ.conj() - np.eye(Nlayer, dtype=np.complex128)
        spectra[trial] = evals
    return covariances, spectra


def cavg_top_layer_pm1_covariance(G_pm1, model):
    Nlayer = 2 * model.Nx * model.Ny
    G_arr = np.asarray(G_pm1, dtype=np.complex128)
    if G_arr.shape == (model.Ntot, model.Ntot):
        return G_arr[:Nlayer, :Nlayer]
    if G_arr.shape == (Nlayer, Nlayer):
        return G_arr
    raise ValueError(f"unexpected covariance shape {G_arr.shape}; expected ({Nlayer},{Nlayer}) or ({model.Ntot},{model.Ntot})")


def cavg_squared_corr_profile_from_pm1_covariance(G_pm1, model, x0, ry_vals):
    Nx, Ny = model.Nx, model.Ny
    Nlayer = 2 * Nx * Ny
    G_top = cavg_top_layer_pm1_covariance(G_pm1, model)
    G2 = 0.5 * (G_top + np.eye(Nlayer, dtype=np.complex128))
    G6 = G2.reshape(2, Nx, Ny, 2, Nx, Ny, order="F")
    Gker = np.transpose(G6, (1, 2, 0, 4, 5, 3))

    x0 = int(x0) % Nx
    ry_arr = np.atleast_1d(ry_vals).astype(int)
    Gx = Gker[x0, :, :, x0, :, :]
    Y = np.arange(Ny, dtype=np.intp)[:, None]
    Yp = (Y + ry_arr[None, :]) % Ny
    Gx_re = np.transpose(Gx, (0, 2, 1, 3)).reshape(Ny * Ny, 2, 2)
    flat_ix = (Y * Ny + Yp).reshape(-1)
    blocks = Gx_re[flat_ix].reshape(Ny, ry_arr.size, 2, 2)
    return np.sum(np.abs(blocks) ** 2, axis=(0, 2, 3)) / (2.0 * Ny)


def cavg_profile_and_fit(G_pm1, model, *, fit_ry_min=None, fit_ry_max=None):
    fit_ry_min = CAVG_FIT_RY_MIN if fit_ry_min is None else int(fit_ry_min)
    fit_ry_max = CAVG_FIT_RY_MAX if fit_ry_max is None else int(fit_ry_max)
    Nx, Ny = model.Nx, model.Ny
    ry_vals = np.arange(1, Ny // 2, dtype=int)
    if ry_vals.size == 0:
        raise ValueError("no positive r_y values available")
    corr_all_x = np.stack([
        cavg_squared_corr_profile_from_pm1_covariance(G_pm1, model, x0, ry_vals).real
        for x0 in range(Nx)
    ], axis=0)
    C_avg = np.mean(corr_all_x, axis=0)
    C_avg_clip = np.clip(C_avg, 1e-300, None)
    log_chord = np.log((Ny / np.pi) * np.sin(np.pi * ry_vals / Ny))
    log_ry = np.log(ry_vals.astype(float))
    fit_mask = (ry_vals >= fit_ry_min) & (ry_vals <= fit_ry_max)
    if np.count_nonzero(fit_mask) < 3:
        return {
            "ry_vals": ry_vals,
            "C_avg": C_avg,
            "log_ry": log_ry,
            "log_chord": log_chord,
            "fit_mask": fit_mask,
            "slope": np.nan,
            "intercept": np.nan,
            "r2": np.nan,
            "fit_line": np.full_like(log_chord, np.nan, dtype=float),
        }
    x_fit = log_chord[fit_mask]
    y_fit = np.log(C_avg_clip[fit_mask])
    slope, intercept = np.polyfit(x_fit, y_fit, 1)
    y_hat = slope * x_fit + intercept
    ss_res = float(np.sum((y_fit - y_hat) ** 2))
    ss_tot = float(np.sum((y_fit - np.mean(y_fit)) ** 2))
    r2 = np.nan if ss_tot == 0.0 else 1.0 - ss_res / ss_tot
    return {
        "ry_vals": ry_vals,
        "C_avg": C_avg,
        "log_ry": log_ry,
        "log_chord": log_chord,
        "fit_mask": fit_mask,
        "slope": float(slope),
        "intercept": float(intercept),
        "r2": float(r2),
        "fit_line": slope * log_chord + intercept,
    }




def cavg_fixed_x_profile_and_fit(G_pm1, model, x0, *, fit_ry_min=None, fit_ry_max=None):
    fit_ry_min = CAVG_FIT_RY_MIN if fit_ry_min is None else int(fit_ry_min)
    fit_ry_max = CAVG_FIT_RY_MAX if fit_ry_max is None else int(fit_ry_max)
    Ny = model.Ny
    ry_vals = np.arange(1, Ny // 2, dtype=int)
    if ry_vals.size == 0:
        raise ValueError("no positive r_y values available")
    C_vec = cavg_squared_corr_profile_from_pm1_covariance(G_pm1, model, int(x0), ry_vals).real
    C_vec_clip = np.clip(C_vec, 1e-300, None)
    log_chord = np.log((Ny / np.pi) * np.sin(np.pi * ry_vals / Ny))
    log_ry = np.log(ry_vals.astype(float))
    fit_mask = (ry_vals >= fit_ry_min) & (ry_vals <= fit_ry_max)
    if np.count_nonzero(fit_mask) < 3:
        return {
            "ry_vals": ry_vals,
            "C_avg": C_vec,
            "log_ry": log_ry,
            "log_chord": log_chord,
            "fit_mask": fit_mask,
            "slope": np.nan,
            "intercept": np.nan,
            "r2": np.nan,
            "fit_line": np.full_like(log_chord, np.nan, dtype=float),
        }
    x_fit = log_chord[fit_mask]
    y_fit = np.log(C_vec_clip[fit_mask])
    slope, intercept = np.polyfit(x_fit, y_fit, 1)
    y_hat = slope * x_fit + intercept
    ss_res = float(np.sum((y_fit - y_hat) ** 2))
    ss_tot = float(np.sum((y_fit - np.mean(y_fit)) ** 2))
    r2 = np.nan if ss_tot == 0.0 else 1.0 - ss_res / ss_tot
    return {
        "ry_vals": ry_vals,
        "C_avg": C_vec,
        "log_ry": log_ry,
        "log_chord": log_chord,
        "fit_mask": fit_mask,
        "slope": float(slope),
        "intercept": float(intercept),
        "r2": float(r2),
        "fit_line": slope * log_chord + intercept,
    }

def cavg_record_from_fit(*, fit, case_spec, nshell, variant, run_name, state_type, trial_orbital, triv_region_local_mode, dw_truncation, model, observable="xavg", fixed_x=None, dw_index=None):
    slope = float(fit["slope"])
    percent_error = np.nan if not np.isfinite(slope) else 100.0 * abs(slope - CAVG_TARGET_SLOPE) / abs(CAVG_TARGET_SLOPE)
    return {
        "percent_error": percent_error,
        "slope": slope,
        "target_slope": CAVG_TARGET_SLOPE,
        "r2": float(fit["r2"]),
        "intercept": float(fit["intercept"]),
        "run_name": run_name,
        "state_type": state_type,
        "trial_orbital": trial_orbital,
        "observable": observable,
        "fixed_x": np.nan if fixed_x is None else int(fixed_x),
        "dw_index": np.nan if dw_index is None else int(dw_index),
        "case_key": case_spec["key"],
        "case_name": case_spec["name"],
        "DW": bool(case_spec["DW"]),
        "alpha_1": case_spec["alpha_1"],
        "alpha_2": case_spec["alpha_2"],
        "variant": variant,
        "nshell": "None" if nshell is None else str(int(nshell)),
        "triv_region_local_mode": bool(triv_region_local_mode),
        "dw_truncation": bool(dw_truncation),
        "fit_ry_min": int(CAVG_FIT_RY_MIN),
        "fit_ry_max": int(CAVG_FIT_RY_MAX),
        "Nx": int(model.Nx),
        "Ny": int(model.Ny),
    }


def cavg_plot_profile(fit, *, title):
    fig, axes = plt.subplots(1, 2, figsize=(13.8, 4.8), constrained_layout=True)
    axes[0].plot(fit["log_ry"], np.log(np.clip(fit["C_avg"], 1e-300, None)), marker="o", ms=3.5, lw=1.2)
    axes[0].set_xlabel(r"$\log r_y$")
    axes[0].set_ylabel(r"$\log C_{\mathrm{avg}}(r_y)$")
    axes[0].set_title(r"$\log C_{\mathrm{avg}}$ vs $\log r_y$")
    axes[0].grid(alpha=0.3)

    axes[1].plot(fit["log_chord"], np.log(np.clip(fit["C_avg"], 1e-300, None)), marker="o", ms=3.5, lw=1.2, label="data")
    axes[1].plot(fit["log_chord"], fit["fit_line"], ls="--", lw=1.8, label=rf"fit: $m={fit['slope']:.4f}$, $R^2={fit['r2']:.4f}$")
    axes[1].set_xlabel(r"$\log[(N_y/\pi)\sin(\pi r_y/N_y)]$")
    axes[1].set_ylabel(r"$\log C_{\mathrm{avg}}(r_y)$")
    axes[1].set_title("log-chord fit")
    axes[1].grid(alpha=0.3)
    axes[1].legend(fontsize=9)
    fig.suptitle(title)
    plt.show()


def cavg_characterize_configuration(case_spec, *, nshell, variant_spec):
    model = classA_U1FGTN_cavg(
        case_spec["Nx"],
        case_spec["Ny"],
        nshell=nshell,
        DW=case_spec["DW"],
        alpha_1=case_spec["alpha_1"],
        alpha_2=case_spec["alpha_2"],
    )
    triv_region_local_mode = bool(variant_spec["triv_region_local_mode"])
    dw_truncation = bool(variant_spec["dw_truncation"])
    variant = variant_spec["variant"]

    exact_cov = model.G_CI_domain_wall(
        periodic=CAVG_PERIODIC,
        triv_region_local_mode=triv_region_local_mode,
    )
    ow_covariances, ow_spectra = cavg_build_flattened_covariances(
        model,
        dw_truncation=dw_truncation,
        triv_region_local_mode=triv_region_local_mode,
    )

    records = []
    fits = {}
    all_covariances = {"exact": exact_cov, **ow_covariances}
    dw_locations = []
    if bool(case_spec["DW"]) and hasattr(model, "DW_loc") and isinstance(model.DW_loc, (list, tuple)) and len(model.DW_loc) == 2:
        dw_locations = [int(model.DW_loc[0]) % model.Nx, int(model.DW_loc[1]) % model.Nx]

    for run_name, cov in all_covariances.items():
        state_type = "equilibrium" if run_name == "exact" else "ow_flattened"
        trial_orbital = None if run_name == "exact" else run_name

        fit = cavg_profile_and_fit(cov, model)
        fits[(run_name, "xavg", None)] = fit
        records.append(cavg_record_from_fit(
            fit=fit,
            case_spec=case_spec,
            nshell=nshell,
            variant=variant,
            run_name=run_name,
            state_type=state_type,
            trial_orbital=trial_orbital,
            triv_region_local_mode=triv_region_local_mode,
            dw_truncation=dw_truncation,
            model=model,
            observable="xavg",
        ))
        if CAVG_PLOT_PROFILES:
            cavg_plot_profile(
                fit,
                title=f"{cavg_case_title(case_spec)} | {variant} | nshell={nshell} | {run_name} | xavg",
            )

        for dw_index, x_dw in enumerate(dw_locations):
            fixed_fit = cavg_fixed_x_profile_and_fit(cov, model, x_dw)
            fits[(run_name, "fixed_dw", int(x_dw))] = fixed_fit
            records.append(cavg_record_from_fit(
                fit=fixed_fit,
                case_spec=case_spec,
                nshell=nshell,
                variant=variant,
                run_name=run_name,
                state_type=state_type,
                trial_orbital=trial_orbital,
                triv_region_local_mode=triv_region_local_mode,
                dw_truncation=dw_truncation,
                model=model,
                observable="fixed_dw",
                fixed_x=int(x_dw),
                dw_index=int(dw_index),
            ))
            if CAVG_PLOT_PROFILES:
                cavg_plot_profile(
                    fixed_fit,
                    title=f"{cavg_case_title(case_spec)} | {variant} | nshell={nshell} | {run_name} | x={x_dw}",
                )
    return records, fits


def cavg_run_all_characterizations():
    all_records = []
    all_fits = {}
    for case_spec in CAVG_CASE_SPECS:
        variants = CAVG_DW_VARIANTS if case_spec["DW"] else CAVG_UNIFORM_VARIANTS
        for variant_spec in variants:
            for nshell in CAVG_NSHELL_VALUES:
                key = (case_spec["key"], variant_spec["variant"], "None" if nshell is None else str(int(nshell)))
                print(f"C_avg sweep: case={key[0]}, variant={key[1]}, nshell={key[2]}")
                records, fits = cavg_characterize_configuration(case_spec, nshell=nshell, variant_spec=variant_spec)
                all_records.extend(records)
                all_fits[key] = fits
    df = pd.DataFrame(all_records)
    first_cols = [
        "percent_error",
        "slope",
        "target_slope",
        "r2",
        "run_name",
        "state_type",
        "trial_orbital",
        "observable",
        "fixed_x",
        "dw_index",
        "case_key",
        "case_name",
        "variant",
        "nshell",
        "triv_region_local_mode",
        "dw_truncation",
    ]
    remaining = [col for col in df.columns if col not in first_cols]
    df = df[first_cols + remaining].sort_values(
        ["percent_error", "case_key", "variant", "nshell", "observable", "fixed_x", "run_name"],
        na_position="last",
    ).reset_index(drop=True)
    return df, all_fits


In [ ]:

# Run the standalone C_avg sweep and return the full dataframe.
cavg_fit_df, cavg_fit_data = cavg_run_all_characterizations()
cavg_fit_df


In [ ]:

# Compact grouped view of C_avg slope errors.
cavg_fit_summary_df = (
    cavg_fit_df
    .groupby(["case_key", "variant", "nshell", "observable", "fixed_x", "run_name"], dropna=False)
    .agg(
        percent_error=("percent_error", "mean"),
        slope=("slope", "mean"),
        r2=("r2", "mean"),
        n=("slope", "count"),
    )
    .reset_index()
    .sort_values("percent_error", na_position="last")
)
cavg_fit_summary_df


## Unified Fit Results

Build one long-form fit table from the raw entanglement-contour and square-correlator fit rows. This avoids mixing two incompatible summary schemas.

In [ ]:
def _normalize_nshell_value(v):
    if pd.isna(v):
        return "None"
    if isinstance(v, str):
        text = v.strip()
        if text.lower() in {"", "none", "nan"}:
            return "None"
        try:
            f = float(text)
            return str(int(f)) if f.is_integer() else text
        except ValueError:
            return text
    f = float(v)
    return str(int(f)) if f.is_integer() else str(v)


def _normalize_nshell_column(df):
    out = df.copy()
    if "nshell" in out.columns:
        out["nshell"] = out["nshell"].map(_normalize_nshell_value)
    return out


def _shared_configuration_label(triv_region_local_mode, dw_truncation):
    triv_local = False if pd.isna(triv_region_local_mode) else bool(triv_region_local_mode)
    dw_trunc = False if pd.isna(dw_truncation) else bool(dw_truncation)
    if dw_trunc and triv_local:
        return "dw_truncation_and_triv_region_local_mode"
    if dw_trunc:
        return "dw_truncation"
    if triv_local:
        return "triv_region_local_mode"
    return "standard"


def build_unified_fit_dataframe(slope_df, cavg_df):
    records = []

    if slope_df is not None and not slope_df.empty:
        ent = _normalize_nshell_column(slope_df).copy()
        ent["slope_error_percent"] = ent["percent_diff"]
        ent["analysis_type"] = "entanglement_contour"
        ent["configuration"] = ent["configuration_label"]
        ent["observable"] = ent["x_interval_kind"].astype(str) + "_" + ent["branch"].astype(str)
        records.append(ent.dropna(axis=1, how="all"))

    if cavg_df is not None and not cavg_df.empty:
        corr = _normalize_nshell_column(cavg_df).copy()
        corr["slope_error_percent"] = corr["percent_error"]
        corr["analysis_type"] = "square_correlator"
        corr["configuration"] = corr["variant"]
        corr["branch"] = pd.NA
        corr["x_interval"] = pd.NA
        corr["x_interval_kind"] = pd.NA
        corr["x_interval_start"] = pd.NA
        corr["x_interval_stop"] = pd.NA
        corr["slope_err"] = pd.NA
        records.append(corr.dropna(axis=1, how="all"))

    if not records:
        return pd.DataFrame()

    unified = pd.concat(records, ignore_index=True, sort=False)
    unified = unified.drop(columns=[c for c in ["percent_diff", "percent_error", "configuration_label", "variant"] if c in unified.columns])
    if {"triv_region_local_mode", "dw_truncation"}.issubset(unified.columns):
        unified["shared_configuration"] = [
            _shared_configuration_label(triv_local, dw_trunc)
            for triv_local, dw_trunc in zip(unified["triv_region_local_mode"], unified["dw_truncation"])
        ]

    first_cols = [
        "slope_error_percent",
        "analysis_type",
        "observable",
        "case_key",
        "shared_configuration",
        "configuration",
        "nshell",
        "run_name",
        "state_type",
        "trial_orbital",
        "slope",
        "target_slope",
        "r2",
    ]
    context_cols = [
        "branch",
        "x_interval",
        "x_interval_kind",
        "x_interval_start",
        "x_interval_stop",
        "slope_err",
        "fixed_x",
        "dw_index",
        "fit_ry_min",
        "fit_ry_max",
        "intercept",
        "case_name",
        "triv_region_local_mode",
        "dw_truncation",
        "DW",
        "alpha_1",
        "alpha_2",
        "Nx",
        "Ny",
    ]
    ordered_cols = [c for c in first_cols + context_cols if c in unified.columns]
    remaining_cols = [c for c in unified.columns if c not in ordered_cols]
    unified = unified[ordered_cols + remaining_cols]
    return unified.sort_values(
        ["slope_error_percent", "analysis_type", "case_key", "configuration", "nshell", "run_name", "observable"],
        na_position="last",
    ).reset_index(drop=True)


def build_unified_fit_summary_dataframe(unified_df):
    if unified_df is None or unified_df.empty:
        return pd.DataFrame()

    group_cols = [
        c for c in [
            "case_key",
            "shared_configuration",
            "nshell",
            "run_name",
            "state_type",
            "trial_orbital",
            "case_name",
            "triv_region_local_mode",
            "dw_truncation",
            "DW",
            "alpha_1",
            "alpha_2",
            "Nx",
            "Ny",
        ]
        if c in unified_df.columns
    ]

    analysis_label_map = {
        "entanglement_contour": "EE",
        "square_correlator": "CS",
    }

    metric_map = {
        "mean slope percent error": ("slope_error_percent", "mean"),
        "max slope percent error": ("slope_error_percent", "max"),
        "mean slope": ("slope", "mean"),
        "mean R2": ("r2", "mean"),
        "n fits": ("slope", "count"),
    }

    pieces = []
    for analysis_type, suffix in analysis_label_map.items():
        subset = unified_df[unified_df["analysis_type"] == analysis_type]
        if subset.empty:
            continue
        agg = (
            subset
            .groupby(group_cols, dropna=False)
            .agg(**{label: spec for label, spec in metric_map.items()})
            .reset_index()
        )
        if "configuration" in subset.columns:
            config_lookup = (
                subset[group_cols + ["configuration"]]
                .drop_duplicates()
                .groupby(group_cols, dropna=False)["configuration"]
                .agg(lambda vals: " | ".join(sorted({str(v) for v in vals if pd.notna(v)})))
                .reset_index(name=f"configuration ({suffix})")
            )
            agg = agg.merge(config_lookup, on=group_cols, how="left")
        rename_cols = {label: f"{label} ({suffix})" for label in metric_map.keys()}
        agg = agg.rename(columns=rename_cols)
        pieces.append(agg)

    if not pieces:
        return pd.DataFrame(columns=group_cols)

    summary = pieces[0]
    for piece in pieces[1:]:
        summary = summary.merge(piece, on=group_cols, how="outer")
    if {"mean slope percent error (EE)", "mean slope percent error (CS)"}.issubset(summary.columns):
        summary["mean slope percent error delta (CS-EE)"] = (
            summary["mean slope percent error (CS)"] - summary["mean slope percent error (EE)"]
        )
    if {"max slope percent error (EE)", "max slope percent error (CS)"}.issubset(summary.columns):
        summary["max slope percent error delta (CS-EE)"] = (
            summary["max slope percent error (CS)"] - summary["max slope percent error (EE)"]
        )

    metric_order = []
    for suffix in ["EE", "CS"]:
        metric_order.extend([
            f"mean slope percent error ({suffix})",
            f"max slope percent error ({suffix})",
            f"mean slope ({suffix})",
            f"mean R2 ({suffix})",
            f"n fits ({suffix})",
        ])

    comparison_cols = [
        c for c in [
            "configuration (EE)",
            "configuration (CS)",
            "mean slope percent error delta (CS-EE)",
            "max slope percent error delta (CS-EE)",
        ]
        if c in summary.columns
    ]
    first_cols = [c for c in group_cols + comparison_cols + metric_order if c in summary.columns]
    remaining_cols = [c for c in summary.columns if c not in first_cols]
    summary = summary[first_cols + remaining_cols]

    sort_cols = [c for c in ["case_key", "shared_configuration", "nshell", "run_name", "mean slope percent error (EE)", "mean slope percent error (CS)"] if c in summary.columns]
    if sort_cols:
        summary = summary.sort_values(sort_cols, na_position="last")
    return summary.reset_index(drop=True)


unified_fit_df = build_unified_fit_dataframe(slope_fit_df, cavg_fit_df)
unified_fit_summary_df = build_unified_fit_summary_dataframe(unified_fit_df)

print(f"unified_fit_df shape = {unified_fit_df.shape}")
print(f"unified_fit_summary_df shape = {unified_fit_summary_df.shape}")
print("missing configuration rows:", int(unified_fit_df["configuration"].isna().sum()) if "configuration" in unified_fit_df else 0)
unified_fit_summary_df


## Dataframe Exports

Save analysis tables for later use. CSV and pickle are guaranteed outputs; Parquet is attempted as a best-effort optional export because this environment can hit a pandas/pyarrow extension-type issue.

In [ ]:

DATAFRAME_OUTPUT_DIR = NOTEBOOK_DATA_DIR
DATAFRAME_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def _save_dataframe_outputs(df, stem, output_dir=DATAFRAME_OUTPUT_DIR):
    """Save a dataframe as CSV and pickle, with best-effort Parquet."""
    if df is None:
        return {"name": stem, "status": "missing"}
    df_to_save = df.copy()
    csv_path = output_dir / f"{stem}.csv"
    pkl_path = output_dir / f"{stem}.pkl"
    parquet_path = output_dir / f"{stem}.parquet"

    record = {
        "name": stem,
        "rows": int(len(df_to_save)),
        "columns": int(len(df_to_save.columns)),
        "csv_path": str(csv_path),
        "pkl_path": str(pkl_path),
        "parquet_path": str(parquet_path),
    }

    df_to_save.to_csv(csv_path, index=False)
    df_to_save.to_pickle(pkl_path)
    record["csv_status"] = "saved"
    record["pkl_status"] = "saved"

    try:
        df_to_save.to_parquet(parquet_path, index=False)
        record["parquet_status"] = "saved"
    except Exception as exc:
        record["parquet_status"] = f"skipped: {type(exc).__name__}: {exc}"
    return record


dataframes_to_save = {
    "unified_fit_df": globals().get("unified_fit_df"),
    "unified_fit_summary_df": globals().get("unified_fit_summary_df"),
    "flattened_chern_df": globals().get("flattened_chern_df"),
    "slope_fit_df": globals().get("slope_fit_df"),
    "slope_fit_summary_df": globals().get("slope_fit_summary_df"),
    "cavg_fit_df": globals().get("cavg_fit_df"),
    "cavg_fit_summary_df": globals().get("cavg_fit_summary_df"),
}

saved_dataframe_manifest = pd.DataFrame([
    _save_dataframe_outputs(df, stem)
    for stem, df in dataframes_to_save.items()
])

manifest_path = DATAFRAME_OUTPUT_DIR / "dataframe_manifest.csv"
saved_dataframe_manifest.to_csv(manifest_path, index=False)
print(f"Saved dataframe manifest to {manifest_path}")

if "unified_fit_df" in globals() and not unified_fit_df.empty:
    assert "percent_diff" not in unified_fit_df.columns
    assert "percent_error" not in unified_fit_df.columns
    assert unified_fit_df.columns[0] == "slope_error_percent"
    assert set(["entanglement_contour", "square_correlator"]).issubset(set(unified_fit_df["analysis_type"].unique()))
    assert unified_fit_df["configuration"].notna().all()
if "unified_fit_summary_df" in globals() and not unified_fit_summary_df.empty:
    assert "analysis_type" not in unified_fit_summary_df.columns
    assert unified_fit_summary_df.columns[0] in {"mean slope percent error (EE)", "mean slope percent error (CS)"}

saved_dataframe_manifest
